In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
            .appName("FinalAssignment2")\
            .getOrCreate()

## 1. Basic DataFrame Operations 
1. Load the sales.csv and customer.csv files into separate DataFrames. 
2. Display the schema of both DataFrames. 
3. Show the first 5 rows from the sales DataFrame. 
4. Count the number of rows and columns in the customer DataFrame. 

In [2]:
# Loading csv files
# NOTE: here we are using inferSchema as there is no complex schema there
sales_df = spark.read.option("header",True).option("inferschema",True).csv("sales.txt")
customer_df = spark.read.option("header",True).option("inferschema",True).csv("customer.txt")

sales_df_copy = sales_df
customer_df_copy = customer_df

print("sales.txt")
sales_df.show()
print("customer.txt")
customer_df.show()

print(" |-- sales schema")
sales_df.printSchema()
print(" |-- customer schema")
customer_df.printSchema()

# showing first 5 rows from sales df
print("First 5 rows from sales_df")
sales_df.show(5)

total_rows = customer_df.count()
total_cols = len(customer_df.columns)
print(f'\nNo. of Columns in customer_df: {total_cols}')
print(f'No. of rows in customer_df: {total_rows}')

sales.txt
+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       6|        101| Mobile| 15000|2023-05-10| South|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|      10|        105| Laptop| 70000|2023-09-25| North|
+--------+-----------+-------+------+----------+------+

customer.txt
+-----------+-------------+--------------------+---+---------+
|customer_id|customer_name|               email|age|     city|
+-----------+-------------+--------------------+---+---------+
|  

## 2. Data Cleaning 
5. Remove duplicate rows from the sales DataFrame based on 
customer_id,product,amount,sale_date,region columns 
6. Drop rows where any column in the customer DataFrame has null values. 
7. Replace null values in the amount column of the sales DataFrame with 0. 
8. Replace null values in the email column of the customer DataFrame with the value 
"unknown". 

In [3]:
# removing duplicate rows from sales DF
total_rows = sales_df.count()
print(f'no. of rows before droping duplicates: {total_rows}')

sales_df = sales_df.dropDuplicates(["customer_id","product","amount","sale_date","region"])

total_rows = sales_df.count()
print(f'no. of rows after droping duplicates: {total_rows}')

no. of rows before droping duplicates: 10
no. of rows after droping duplicates: 10


In [4]:
# droping rows where any column in customer df has null values
total_rows = customer_df.count()
print(f'no. of rows before droping nulls: {total_rows}')

customer_df = customer_df.na.drop()

total_rows = customer_df.count()
print(f'no. of rows after droping nulls: {total_rows}')

no. of rows before droping nulls: 7
no. of rows after droping nulls: 7


In [5]:
# Replace null values in the amount column of the sales DataFrame with 0.  
sales_df = sales_df.na.fill(0)
print("null amounts replaced with 0 successfully")

null amounts replaced with 0 successfully


In [6]:
# Replace null values in the email column of the customer DataFrame with the value "unknown". 
customer_df = customer_df.na.fill("unknown")
print('null emails replaced with \"unknown\" successfully')

null emails replaced with "unknown" successfully


## 3. Column Manipulation 
9. Add a new column discounted_amount to the sales DataFrame that applies a 10% 
discount on amount. 
10. Rename the city column in the customer DataFrame to customer_city. 
11. Drop the region column from the sales DataFrame. 
12. Create a new column customer_age_category in the customer DataFrame based on age: 
    a. "Youth" for age < 30 
    b. "Adult" for 30 <= age < 50 
    c. "Senior" for age >= 50 

In [7]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [8]:
# Add a new column discounted_amount to the sales DataFrame that applies a 10% discount on amount. 
sales_df.show()

# not changing actual schema
_sales_df = sales_df.withColumn("discounted_amount",col("amount") * 0.1)
_sales_df.show()


+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|      10|        105| Laptop| 70000|2023-09-25| North|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|       6|        101| Mobile| 15000|2023-05-10| South|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       4|        104| Laptop| 55000|2023-03-15|  East|
+--------+-----------+-------+------+----------+------+

+--------+-----------+-------+------+----------+------+-----------------+
|sales_id|customer_id|product|amount| sale_date|region|discounted_amount|
+--------+-----------+-------+------+----------+------+------------

In [9]:
# Rename the city column in the customer DataFrame to customer_city. 

customer_df = customer_df.withColumnRenamed("city","customer_city")

customer_df.show()

+-----------+-------------+--------------------+---+-------------+
|customer_id|customer_name|               email|age|customer_city|
+-----------+-------------+--------------------+---+-------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|
|        102|  Meena Verma|meena.verma@email...| 34|       Mumbai|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|
|        106|   Vikas Jain|vikas.jain@email.com| 31|      Chennai|
|        107|     Amit Roy|  amit.roy@email.com| 35|      Kolkata|
+-----------+-------------+--------------------+---+-------------+



In [ ]:
# droping region column
# NOTE: region column is used later in some queries, so we are not changing actual DF 

_sales_df = sales_df.drop("region")
print('region column dropped successfully')

region column dropped successfully


In [27]:
# adding new column customer_age_category
customer_df = customer_df.withColumn("customer_age_category",
                       when(col("age") < 30, "Youth")\
                       .when((col("age") >= 30) & (col("age") < 50) ,"Adult")\
                       .when((col("age") >= 50), "Senior")
                       )
customer_df.show()
print('column customer_age_category added successfully')

+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+-----------+-------------+--------------------+---+-------------+---------------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        102|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|                Youth|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|                Youth|
|        106|   Vikas Jain|vikas.jain@email.com| 31|      Chennai|                Adult|
|        107|     Amit Roy|  amit.roy@email.com| 35|      Kolkata|                Adult|
+-----------+-------------+--------------------+---+-------------+---------------------+

column customer_age_

## 4. Filtering 
13. Filter the sales DataFrame to show only rows where amount is greater than 50,000. 
14. Filter the customer DataFrame to show customers aged between 25 and 30. 
15. Identify all customers who have made purchases in more than one region. 
16. Filter the top 3 sales based on amount for each product.

In [ ]:
sales_df.filter(col("amount") > 50000).show()

customer_df.filter(col("age").between(25,30)).show()

# TODO: join with customer_df to display customer_name
sales_df.groupBy("customer_id")\
    .agg(count_distinct(col("region"))\
    .alias("count_of_regions"))\
    .filter(col("count_of_regions") > 1)\
    .show()

sales_df.select(
    col("*"),
    row_number().over(Window.partitionBy(col("product")).orderBy(desc(col("amount")))).alias("row_number")
).filter(col("row_number") <= 3).show()

Q13: Sales where amount > 50,000
+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       7|        102| Laptop| 60000|2023-06-15|  East|
|      10|        105| Laptop| 70000|2023-09-25| North|
|       4|        104| Laptop| 55000|2023-03-15|  East|
+--------+-----------+-------+------+----------+------+

Q14: Customers aged between 25 and 30
+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+-----------+-------------+--------------------+---+-------------+---------------------+
|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|        104|  Priya Patel|priya.patel@email...| 27|    Ahmedabad|                Youth|
| 

## 5. Joins 
17. Perform an inner join between sales and customer DataFrames on customer_id. 
18. Perform a left join to include all records from sales and matching records from 
customer. 
19. Perform a full outer join between sales and customer DataFrames. 
20. Identify customers who have not made any purchases by performing an anti-join.

In [13]:
# joining both DFs based on customer_id
print("Inner-Join")
sales_df.join(
    customer_df,
    (sales_df.customer_id == customer_df.customer_id),
    "inner"
).show()

# left join both DFs based on customer_id
print("Left-Join")
sales_df.join(
    customer_df,
    (sales_df.customer_id == customer_df.customer_id),
    "left"
).show()

# fullouter join both DFs based on customer_id
print("Full-Join")
sales_df.join(
    customer_df,
    (sales_df.customer_id == customer_df.customer_id),
    "full"
).show()

# performing anti-join to identify the customers who have not purchased any product
print("Anti-Join")
customer_df.join(
    sales_df,
    (sales_df.customer_id == customer_df.customer_id),
    "anti"
).show()


Inner-Join
+--------+-----------+-------+------+----------+------+-----------+-------------+--------------------+---+-------------+---------------------+
|sales_id|customer_id|product|amount| sale_date|region|customer_id|customer_name|               email|age|customer_city|customer_age_category|
+--------+-----------+-------+------+----------+------+-----------+-------------+--------------------+---+-------------+---------------------+
|       1|        101| Laptop| 50000|2023-01-15| North|        101|  Arun Sharma|arun.sharma@email...| 28|        Delhi|                Youth|
|       3|        103| Tablet| 20000|2023-03-05|  West|        103|  Rahul Yadav|rahul.yadav@email...| 30|    Bangalore|                Adult|
|       7|        102| Laptop| 60000|2023-06-15|  East|        102|  Meena Verma|meena.verma@email...| 34|       Mumbai|                Adult|
|      10|        105| Laptop| 70000|2023-09-25| North|        105|  Sneha Reddy|sneha.reddy@email...| 29|    Hyderabad|           

## 6. Aggregations 
21. Calculate the total sales amount for each product. 
22. Find the average age of customers in the customer DataFrame. 
23. Calculate the maximum and minimum sales amounts in the sales DataFrame. 
24. Group the customer DataFrame by customer_city and count the number of customers in 
each city.

In [14]:
# total sales amount
sales_df.groupBy("product").agg(sum("amount").alias("total_sales_amount")).show()

# average age of customers
customer_df.select(avg(col("age")).alias("avg_age_of_customers")).show()

# max and min sales amount
sales_df.select(
    max(col("amount")).alias("max_sales_amount"),
    min(col("amount")).alias("min_sales_amount"),
).show()

customer_df.groupBy("customer_city")\
    .agg(count(col("customer_id"))\
    .alias("customer_count"))\
    .show()

+-------+------------------+
|product|total_sales_amount|
+-------+------------------+
| Laptop|            235000|
| Mobile|             30000|
| Tablet|             40000|
|Desktop|             85000|
+-------+------------------+

+--------------------+
|avg_age_of_customers|
+--------------------+
|  30.571428571428573|
+--------------------+

+----------------+----------------+
|max_sales_amount|min_sales_amount|
+----------------+----------------+
|           70000|           15000|
+----------------+----------------+

+-------------+--------------+
|customer_city|customer_count|
+-------------+--------------+
|    Bangalore|             1|
|      Chennai|             1|
|       Mumbai|             1|
|    Ahmedabad|             1|
|      Kolkata|             1|
|        Delhi|             1|
|    Hyderabad|             1|
+-------------+--------------+



## 7. Sorting 
25. Sort the sales DataFrame by amount in descending order. 
26. Sort the customer DataFrame by age in ascending order. 

In [15]:
print("sales data sorted by amount in descending order")
sales_df.sort(desc(col("amount"))).show()

print("customer data sorted by age in ascending order")
customer_df.sort(col("age")).show()

sales data sorted by amount in descending order
+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|      10|        105| Laptop| 70000|2023-09-25| North|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|       4|        104| Laptop| 55000|2023-03-15|  East|
|       1|        101| Laptop| 50000|2023-01-15| North|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       6|        101| Mobile| 15000|2023-05-10| South|
+--------+-----------+-------+------+----------+------+

customer data sorted by age in ascending order
+-----------+-------------+--------------------+---+-------------+---------------------+
|customer_id|customer_name|    

## 8. Union Operations 
27. Add a new dataset for customers and perform a union operation with the customer 
DataFrame. 
28. Combine the sales DataFrame with another DataFrame containing additional sales 
records. 

In [16]:
print("Union of old and new customers")
new_customers_df = spark.read\
                        .option("header",True)\
                        .option("inferschema",True)\
                        .csv("new_customers.txt")

customer_df_copy.union(new_customers_df).show()

print("Union of old and new sales")
additional_sales_df = spark.read\
                        .option("header",True)\
                        .option("inferschema",True)\
                        .csv("additional_sales.txt")

sales_df_copy.union(additional_sales_df).show()

Union of old and new customers
+-----------+-------------+--------------------+---+---------+
|customer_id|customer_name|               email|age|     city|
+-----------+-------------+--------------------+---+---------+
|        101|  Arun Sharma|arun.sharma@email...| 28|    Delhi|
|        102|  Meena Verma|meena.verma@email...| 34|   Mumbai|
|        103|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|
|        104|  Priya Patel|priya.patel@email...| 27|Ahmedabad|
|        105|  Sneha Reddy|sneha.reddy@email...| 29|Hyderabad|
|        106|   Vikas Jain|vikas.jain@email.com| 31|  Chennai|
|        107|     Amit Roy|  amit.roy@email.com| 35|  Kolkata|
|        108|  Rahul Yadav|rahul.yadav@email...| 30|Bangalore|
|        109|  Priya Patel|priya.patel@email...| 27|Ahmedabad|
+-----------+-------------+--------------------+---+---------+

Union of old and new sales
+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+

## 9. Window Functions 
29. Rank the sales records based on the amount column. 
30. Add a cumulative sum of amount for each product in the sales DataFrame. 
31. Add a column that calculates the difference between each customer's amount and the 
average amount within their product group. 

In [17]:
sales_df.withColumn(
    "Rank",
    rank().over(Window.orderBy(desc(col("amount"))))
).show()

sales_df.withColumn(
    "cumulative_sum_of_amount",
    sum(col("amount")).over(Window\
               .partitionBy(col("product"))\
                .orderBy(col("sale_date").cast(DateType()))\
                .rowsBetween(Window.unboundedPreceding,Window.currentRow)
            )
).show()

sales_df.withColumn(
    "difference_amount_avg",
    col("amount") - avg("amount").over(Window.partitionBy("customer_id"))
).show()

+--------+-----------+-------+------+----------+------+----+
|sales_id|customer_id|product|amount| sale_date|region|Rank|
+--------+-----------+-------+------+----------+------+----+
|      10|        105| Laptop| 70000|2023-09-25| North|   1|
|       7|        102| Laptop| 60000|2023-06-15|  East|   2|
|       4|        104| Laptop| 55000|2023-03-15|  East|   3|
|       1|        101| Laptop| 50000|2023-01-15| North|   4|
|       9|        104|Desktop| 45000|2023-08-10|  West|   5|
|       5|        105|Desktop| 40000|2023-04-20| North|   6|
|       3|        103| Tablet| 20000|2023-03-05|  West|   7|
|       8|        103| Tablet| 20000|2023-07-05| North|   7|
|       2|        102| Mobile| 15000|2023-02-10| South|   9|
|       6|        101| Mobile| 15000|2023-05-10| South|   9|
+--------+-----------+-------+------+----------+------+----+

+--------+-----------+-------+------+----------+------+------------------------+
|sales_id|customer_id|product|amount| sale_date|region|cumulativ

## 10. Partitioning 
32. Write the sales DataFrame to a partitioned Parquet file by region. 
33. Partition the customer DataFrame by customer_city and save it as a CSV file. 

In [18]:
# Parquet write (partition by region)
try:
    sales_df_copy.write \
        .mode("overwrite") \
        .partitionBy("region") \
        .parquet("parquet-partitions")

    print("Parquet written successfully")

except Exception as e:
    print(f"Parquet write failed: {e}")


# CSV write (partition by customer_id)
try:
    customer_df_copy.write \
        .mode("overwrite") \
        .partitionBy("customer_id") \
        .csv("csv-partitions")

    print("CSV written successfully")

except Exception as e:
    print(f"CSV write failed: {e}")

Parquet write failed: An error occurred while calling o291.parquet.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1068)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1116)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:798)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:838)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:837)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)
	at org.apache.hadoop.fs.RawLo

## 11. Real-World Scenarios 
34. Calculate the percentage contribution of each product to the total sales. 
35. Extract the year from sale_date and group by year to calculate total sales. 
36. Identify the most purchased product in each region. 
37. Add a column to show the difference between the highest and lowest sales for each 
product. 
38. Write the result of the join between sales and customer to parquet file. 
39. Identify products that were sold in the last 6 months. 
40. Calculate the average sales amount per customer

In [19]:
# creating sales view
sales_df.createOrReplaceTempView("sales")
sales_df_copy.createOrReplaceTempView("sales_copy")
sales_df.show()

+--------+-----------+-------+------+----------+------+
|sales_id|customer_id|product|amount| sale_date|region|
+--------+-----------+-------+------+----------+------+
|       1|        101| Laptop| 50000|2023-01-15| North|
|       3|        103| Tablet| 20000|2023-03-05|  West|
|       7|        102| Laptop| 60000|2023-06-15|  East|
|      10|        105| Laptop| 70000|2023-09-25| North|
|       5|        105|Desktop| 40000|2023-04-20| North|
|       2|        102| Mobile| 15000|2023-02-10| South|
|       9|        104|Desktop| 45000|2023-08-10|  West|
|       6|        101| Mobile| 15000|2023-05-10| South|
|       8|        103| Tablet| 20000|2023-07-05| North|
|       4|        104| Laptop| 55000|2023-03-15|  East|
+--------+-----------+-------+------+----------+------+



In [20]:
# Calculate the percentage contribution of each product to the total sales.
query34 = """
select 
    product
    , (sum(amount) * 100.0) / (select sum(amount) from sales) as percent_contribution
from
    sales s
group by 
    product
"""

spark.sql(query34).show()

+-------+--------------------+
|product|percent_contribution|
+-------+--------------------+
| Laptop|   60.25641025641026|
| Mobile|    7.69230769230769|
| Tablet|   10.25641025641026|
|Desktop|   21.79487179487179|
+-------+--------------------+



In [21]:
#  Extract the year from sale_date and group by year to calculate total sales.
query35 = """
select
    year(sale_date) as year
    , sum(amount) as total_sales
from
    sales s
group by
    year

"""
spark.sql(query35).show()

+----+-----------+
|year|total_sales|
+----+-----------+
|2023|     390000|
+----+-----------+



In [22]:
#  Identify the most purchased product in each region. 
query36 = """ 
select distinct
    region,
    first_value(product) over (
        partition by region
        order by amount
    ) as most_purchased_product
from
    sales_copy
"""

spark.sql(query36).show()

+------+----------------------+
|region|most_purchased_product|
+------+----------------------+
|  East|                Laptop|
| North|                Tablet|
| South|                Mobile|
|  West|                Tablet|
+------+----------------------+



In [23]:
# Add a column to show the difference between the highest and lowest sales for each product.
# here we are only showing product column instead of showing all columns of sales
query37 = """
select
    product,
    max(amount) - min(amount) as difference
from
    sales
group by 
    product
"""

spark.sql(query37).show()

+-------+----------+
|product|difference|
+-------+----------+
| Laptop|     20000|
| Mobile|         0|
| Tablet|         0|
|Desktop|      5000|
+-------+----------+



In [24]:
#  Write the result of the join between sales and customer to parquet file.
try:
    sales_df.join(
        customer_df,
        sales_df.customer_id == customer_df.customer_id,
        "inner"
    ).drop(customer_df.customer_id)\
    .write.mode("overwrite").parquet("joined_table_q_38")
    print("Joined DF written to file successfully")
except Exception as e:
    print(f"Error: {e}")

Error: An error occurred while calling o323.parquet.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1068)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1116)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:798)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:838)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:837)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)
	at org.apache.hadoop.fs.RawLocalFileSystem.m

In [25]:
#  Identify products that were sold in the last 6 months.
sales_df.filter(
    col("sale_date") >= add_months(current_date(), -6)
).select(
    col("sales_id"),
    col("product")
).show()

+--------+-------+
|sales_id|product|
+--------+-------+
+--------+-------+



In [26]:
# Calculate the average sales amount per customer. 
query40 = """
select
    customer_id
    , avg(amount) as avg_sales_amount
from 
    sales
group by
    customer_id
order by
    customer_id
"""

spark.sql(query40).show()

+-----------+----------------+
|customer_id|avg_sales_amount|
+-----------+----------------+
|        101|         32500.0|
|        102|         37500.0|
|        103|         20000.0|
|        104|         50000.0|
|        105|         55000.0|
+-----------+----------------+

